In [1]:

from pathlib import Path
import os
import sys
import subprocess
import json
import shutil
import re

# Kaggle paths.
IN_KAGGLE = Path("/kaggle").exists()
KAGGLE_WORKING = Path("/kaggle/working") if IN_KAGGLE else Path("/mnt/data/kaggle_working")
KAGGLE_INPUT = Path("/kaggle/input") if IN_KAGGLE else Path("/mnt/data")

# Repo/data paths theo notebook cũ của bạn.
REPO_DIR = KAGGLE_WORKING / "SinoNom-NLP"
VIET_DATA_DIR = KAGGLE_INPUT / "datasets/quachthanhhmd/sinonom-local/vietnam"

# Output riêng cho pipeline review/correction.
OUTPUT_DIR = KAGGLE_WORKING / "output_task2_dntc_ocr_review"

# Không xóa repo. Chỉ clone nếu chưa có và bạn bật flag.
CLONE_REPO_IF_MISSING = False
GIT_URL = "https://github.com/quachthanhhmd/SinoNom-NLP.git"
GIT_BRANCH = "dev/thongnguyen"

print("IN_KAGGLE      =", IN_KAGGLE)
print("KAGGLE_WORKING =", KAGGLE_WORKING)
print("KAGGLE_INPUT   =", KAGGLE_INPUT)
print("REPO_DIR       =", REPO_DIR)
print("VIET_DATA_DIR  =", VIET_DATA_DIR)
print("OUTPUT_DIR     =", OUTPUT_DIR)

KAGGLE_WORKING.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


IN_KAGGLE      = True
KAGGLE_WORKING = /kaggle/working
KAGGLE_INPUT   = /kaggle/input
REPO_DIR       = /kaggle/working/SinoNom-NLP
VIET_DATA_DIR  = /kaggle/input/datasets/quachthanhhmd/sinonom-local/vietnam
OUTPUT_DIR     = /kaggle/working/output_task2_dntc_ocr_review


In [2]:

# Cài dependency nhẹ. Chạy lại cell này không sao.
packages = [
    "pymupdf",
    "pandas",
    "openpyxl",
    "pillow",
    "tqdm",
    "underthesea",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=False)

if CLONE_REPO_IF_MISSING and not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--branch", GIT_BRANCH, GIT_URL, str(REPO_DIR)],
        check=True,
    )

if REPO_DIR.exists() and str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print("Repo exists:", REPO_DIR.exists())


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 71.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 99.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 74.8 MB/s eta 0:00:00
Repo exists: False


In [3]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Vietnamese PDF extraction + OCR-error review + manual correction applier.

Designed for scanned Vietnamese books such as Dai Nam Nhat Thong Chi where the
PDF already has a noisy text layer: missing accents, broken spacing, wrong chars,
footer/header contamination, and unstable sentence boundaries.

Main workflow:
  1. extract: PDF -> blocks -> paragraphs -> sentences -> review XLSX/CSV/JSONL
  2. review: edit corrected_text_manual / review_status in XLSX
  3. apply: review XLSX -> final corrected sentences and text files

Install:
  pip install pymupdf pandas openpyxl pillow tqdm
  pip install underthesea   # optional, better Vietnamese sentence splitting

Examples:
  python pdf_extract_review_corrector.py extract \
    --pdfs q1.pdf q2.pdf q3.pdf q4.pdf q5.pdf \
    --out output_dntc \
    --render-crops suspicious \
    --review-mode suspicious

  python pdf_extract_review_corrector.py apply \
    --out output_dntc \
    --review output_dntc/review/suspicious_review.xlsx

Notes:
  - Auto correction is deliberately conservative.
  - Every change keeps raw_text, auto_text, rules, reasons, page, bbox, crop_path.
  - For historical/geographical texts, manual review beats aggressive spell models.
"""
from __future__ import annotations

import argparse
import csv
import hashlib
import json
import math
import os
import re
import statistics
import sys
import unicodedata
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

try:
    import fitz  # PyMuPDF
except Exception as exc:  # pragma: no cover
    raise SystemExit("Missing dependency: pymupdf. Install with: pip install pymupdf") from exc

try:
    import pandas as pd
except Exception as exc:  # pragma: no cover
    raise SystemExit("Missing dependency: pandas. Install with: pip install pandas openpyxl") from exc

try:
    from PIL import Image
except Exception:  # pragma: no cover
    Image = None

try:
    from tqdm import tqdm
except Exception:  # pragma: no cover
    def tqdm(x, **kwargs):
        return x

try:
    from underthesea import sent_tokenize as underthesea_sent_tokenize
except Exception:  # pragma: no cover
    underthesea_sent_tokenize = None


# -----------------------------------------------------------------------------
# Constants and regexes
# -----------------------------------------------------------------------------

VIET_LOWER = "aàáảãạăằắẳẵặâầấẩẫậbcdđeèéẻẽẹêềếểễệghiìíỉĩịklmnoòóỏõọôồốổỗộơờớởỡợpqrstuùúủũụưừứửữựvxyỳýỷỹỵ"
VIET_UPPER = VIET_LOWER.upper()
VIET_CHARS = set(VIET_LOWER + VIET_UPPER)
EXPECTED_TEXT_CHARS_RE = re.compile(
    r"^[0-9A-Za-zÀ-ỹ\s\.,;:\?!…\-–—'\"“”‘’\(\)\[\]\{\}/\\&%+*=<>\n\r\t]+$"
)
TOKEN_RE = re.compile(r"[A-Za-zÀ-ỹĐđ]+(?:[-'][A-Za-zÀ-ỹĐđ]+)?|\d+(?:[\.,]\d+)*")
TERMINAL_RE = re.compile(r"[\.\?!…]\s*([\)\]\}”’\"])?$")
HEADING_TERMINAL_RE = re.compile(r"[:：]\s*$")
PAGE_NUMBER_RE = re.compile(r"^\s*[-–—]*\s*\d{1,4}\s*[-–—]*\s*$")
ROMAN_PAGE_RE = re.compile(r"^\s*[ivxlcdmIVXLCDM]{1,8}\s*$")
PDF_FILENAME_RE = re.compile(r"\b[\w\-]*nhat\w*thong\w*chi\w*\.pdf\b", re.I)
JUNK_CHARS_RE = re.compile(r"[\uFFFD\uFFFC\u001c\u001d\u001e\u001f\u200b\u200c\u200d\ufeff\u00ac\u00a6\u20ac\u02c6\u02dc\ufffe\uffff\u00a9\u00ae\ufffd￾©®ˆ€|_~]")
GLUED_NUM_LETTER_RE = re.compile(r"(\d)([A-Za-zÀ-ỹĐđ])|([A-Za-zÀ-ỹĐđ])(\d)")
MULTISPACE_RE = re.compile(r"[ \t]{2,}")
SPACE_BEFORE_PUNCT_RE = re.compile(r"\s+([,.;:!?%\)\]\}])")
NO_SPACE_AFTER_PUNCT_RE = re.compile(r"([,;:!?])([^\s\d\)\]\}\.,;:!?])")
WORD_DOT_WORD_RE = re.compile(r"([A-Za-zÀ-ỹĐđ])\.([A-Za-zÀ-ỹĐđ])")
UPPER_HEADING_RE = re.compile(r"^[A-ZÀ-ỸĐ\s\-\.,:;\(\)0-9]+$")
FOOTNOTE_START_RE = re.compile(r"^\s*\(?\d{1,2}\)\s+")
LIST_ITEM_RE = re.compile(r"^\s*[-–—•*]\s+")

# Abbreviations and historical forms that should not trigger sentence split.
ABBREVIATIONS = [
    "v.v.", "v.v..", "tr.C.N.", "Tr.C.N.", "CN.", "T.C.N.", "P.", "TS.", "PGS.",
]

KNOWN_SECTION_HEADINGS = {
    "PHAN DA": "PHẦN DÃ",
    "PHẦN DA": "PHẦN DÃ",
    "PHẦN DÃ": "PHẦN DÃ",
    "DUNG DAT VA DIEN CACH": "DỰNG ĐẶT VÀ DIÊN CÁCH",
    "DUNG PAT VA DIEN CACH": "DỰNG ĐẶT VÀ DIÊN CÁCH",
    "DỰNG ĐẶT VA DIỄN CÁCH": "DỰNG ĐẶT VÀ DIÊN CÁCH",
    "DỰNG ĐẶT VÀ DIỄN CÁCH": "DỰNG ĐẶT VÀ DIÊN CÁCH",
    "DỰNG ĐẶT VA DIEN CACH": "DỰNG ĐẶT VÀ DIÊN CÁCH",
    "HINH THE": "HÌNH THẾ",
    "HÌNH THẾ": "HÌNH THẾ",
    "KHI HAU": "KHÍ HẬU",
    "KHÍ HẬU": "KHÍ HẬU",
    "PHONG TUC": "PHONG TỤC",
    "PHONG TỤC": "PHONG TỤC",
    "THANH TRI": "THÀNH TRÌ",
    "THÀNH TRÌ": "THÀNH TRÌ",
    "TRUONG HOC": "TRƯỜNG HỌC",
    "TRƯỜNG HỌC": "TRƯỜNG HỌC",
    "HO KHAU": "HỘ KHẨU",
    "HỘ KHẨU": "HỘ KHẨU",
    "THUE RUONG": "THUẾ RUỘNG",
    "THUẾ RUỘNG": "THUẾ RUỘNG",
    "NUI SONG": "NÚI SÔNG",
    "NÚI SÔNG": "NÚI SÔNG",
    "SON XUYEN": "SƠN XUYÊN",
    "QUAN TAN": "QUAN TẤN",
    "DICH TRAM": "DỊCH TRẠM",
    "THI LAP": "THỊ LẬP",
    "TU MIEU": "TỪ MIẾU",
    "TU QUAN": "TỰ QUÁN",
    "THO SAN": "THỔ SẢN",
    "NHAN VAT": "NHÂN VẬT",
    "DI TICH": "DI TÍCH",
    "PHU HUYEN": "PHỦ HUYỆN",
}

# Conservative auto-correction rules. Add project-specific rules in custom CSV.
# Tuple format: (scope, regex_pattern, replacement, rule_name).
# scope: all | heading | body
BUILTIN_RULES: List[Tuple[str, str, str, str]] = [
    # Titles and headers.
    ("all", r"\bQUOC\s+SU\s+QUAN\s+TRIEU\s+NGUYEN\b", "QUỐC SỬ QUÁN TRIỀU NGUYỄN", "title_quoc_su_quan_ascii"),
    ("all", r"\bQUỐC\s+SU\s+QUAN\s+TRIEU\s+NGUYEN\b", "QUỐC SỬ QUÁN TRIỀU NGUYỄN", "title_quoc_su_quan_mix"),
    ("all", r"\bQUỐC\s+SỬ\s+QUAN\s+TRIỀU\s+NGUYEN\b", "QUỐC SỬ QUÁN TRIỀU NGUYỄN", "title_quoc_su_quan_nguyen"),
    ("all", r"\bVIEN\s+KHOA\s+HOC\s+XA\s+HOI\s+VIET\s+NAM\b", "VIỆN KHOA HỌC XÃ HỘI VIỆT NAM", "title_vien_khxh_ascii"),
    ("all", r"\bVIEN\s+KHOA\s+HỌC\s+XÃ\s+HỘI\s+VIET\s+NAM\b", "VIỆN KHOA HỌC XÃ HỘI VIỆT NAM", "title_vien_khxh_mix"),
    ("all", r"\bVI[ỆE]N\s*S[ỬU]\s*H[ỌO]C\b", "VIỆN SỬ HỌC", "title_vien_su_hoc"),
    ("all", r"\bDAI\s+NAM\s+NH[ẤẬA]T\s+THONG\s+CH[IÍ]\b", "ĐẠI NAM NHẤT THỐNG CHÍ", "title_dai_nam"),
    ("all", r"\bĐẠI\s+NAM\s+NH[ẤẬA]T\s+THONG\s+CH[IÍ]\b", "ĐẠI NAM NHẤT THỐNG CHÍ", "title_dai_nam_thong"),
    ("all", r"\bNH[ẤẬA]T\s+THONG\s+CH[IÍ]\b", "NHẤT THỐNG CHÍ", "title_nhat_thong"),
    ("all", r"\bNH[ÀA]\s+XU[ẤA]T\s+B[ẢA]N\s+THU[ẬA]N\s+H[OÓÒ]A\b", "NHÀ XUẤT BẢN THUẬN HÓA", "title_nxb_thuan_hoa"),

    # Section headings and structural words.
    ("heading", r"\bPHAN\s+D[AÃ]\b", "PHẦN DÃ", "heading_phan_da"),
    ("heading", r"\bD[UƯ]NG\s+[ĐDPI]AT\s+V[AÀ]\s+DI[ÊE]N\s+C[AÁ]CH\b", "DỰNG ĐẶT VÀ DIÊN CÁCH", "heading_dung_dat_dien_cach"),
    ("heading", r"\bHINH\s+THE\b", "HÌNH THẾ", "heading_hinh_the"),
    ("heading", r"\bKHI\s+HAU\b", "KHÍ HẬU", "heading_khi_hau"),
    ("heading", r"\bPHONG\s+TUC\b", "PHONG TỤC", "heading_phong_tuc"),
    ("heading", r"\bTHANH\s+TRI\b", "THÀNH TRÌ", "heading_thanh_tri"),
    ("heading", r"\bTRUONG\s+HOC\b", "TRƯỜNG HỌC", "heading_truong_hoc"),
    ("heading", r"\bHO\s+KHAU\b", "HỘ KHẨU", "heading_ho_khau"),
    ("heading", r"\bTHUE\s+RUONG\b", "THUẾ RUỘNG", "heading_thue_ruong"),
    ("heading", r"\bNUI\s+SONG\b", "NÚI SÔNG", "heading_nui_song"),
    ("heading", r"\bQUYEN\s+([IVXLCDM]+|\d+)\b", r"QUYỂN \1", "heading_quyen"),
    ("heading", r"\bKINH\s+SU\b", "KINH SƯ", "heading_kinh_su"),
    ("heading", r"\bTINH\s+HA\s+TIEN\b", "TỈNH HÀ TIÊN", "heading_tinh_ha_tien"),
    ("heading", r"\bTINH\s+QUANG\s+BINH\b", "TỈNH QUẢNG BÌNH", "heading_tinh_quang_binh"),
    ("heading", r"\bTINH\s+BINH\s+DINH\b", "TỈNH BÌNH ĐỊNH", "heading_tinh_binh_dinh"),
    ("heading", r"\bTINH\s+QUANG\s+YEN\b", "TỈNH QUẢNG YÊN", "heading_tinh_quang_yen"),
    ("heading", r"\bTINH\s+", "TỈNH ", "heading_tinh"),
    ("heading", r"\bPHU\s+", "PHỦ ", "heading_phu"),
    ("heading", r"\bHUYEN\s+", "HUYỆN ", "heading_huyen"),

    # Very common, high-confidence body fixes from these PDFs.
    ("body", r"\bdia\s+giới\b", "địa giới", "body_dia_gioi"),
    ("body", r"\bđia\s+giới\b", "địa giới", "body_dia_gioi2"),
    ("body", r"\bphia\s+", "phía ", "body_phia"),
    ("body", r"\bphía\s+dong\b", "phía đông", "body_phia_dong"),
    ("body", r"\bphía\s+tay\b", "phía tây", "body_phia_tay"),
    ("body", r"\bphía\s+b[ăa]c\b", "phía bắc", "body_phia_bac"),
    ("body", r"\bphía\s+nam\b", "phía nam", "body_phia_nam"),
    ("body", r"\b(\d+)\s*(?:dim|d4m|dam)\b", r"\1 dặm", "body_dam_after_num"),
    ("body", r"\b(\d+)\s*d[ặa]m\b", r"\1 dặm", "body_dam_spacing"),
    ("body", r"\bn[ãă]m\s+(Minh|Gia|Tự|Thiệu|Đồng|Duy|Thành|Long|Canh|Mậu|Nhâm|Giáp|Ất|Bính|Đinh|Kỉ|Kỷ|Tân|Quí|Quý)\b", r"năm \1", "body_nam_context"),
    ("body", r"\bñăm\b", "năm", "body_nam_n_tilde"),
    ("body", r"\bNim\s+", "Năm ", "body_nim"),
    ("body", r"\bHuy[eé]n\b", "Huyện", "body_huyen_cap"),
    ("body", r"\bhuy[eé]n\b", "huyện", "body_huyen_low"),
    ("body", r"\btinh\s+(Quảng|Bình|Hà|An|Gia|Vĩnh|Định|Phú|Nghệ|Thanh|Nam|Bắc|Hải|Lạng|Sơn|Thái|Tuyên|Cao|Kiên|Long)\b", r"tỉnh \1", "body_tinh_place"),
    ("body", r"\bTinh\s+(Quảng|Bình|Hà|An|Gia|Vĩnh|Định|Phú|Nghệ|Thanh|Nam|Bắc|Hải|Lạng|Sơn|Thái|Tuyên|Cao|Kiên|Long)\b", r"Tỉnh \1", "body_tinh_place_cap"),
    ("body", r"\bph[du]\s+(Quảng|Hoài|An|Hải|Sơn|Tân|Qui|Quy|Phú|Quan|Tĩnh|Quảng|Tiên)\b", r"phủ \1", "body_phu_context"),
    ("body", r"\bPh[du]\s+(Quảng|Hoài|An|Hải|Sơn|Tân|Qui|Quy|Phú|Quan|Tĩnh|Quảng|Tiên)\b", r"Phủ \1", "body_phu_context_cap"),
    ("body", r"\bbi€n\b", "biển", "body_bien_euro"),
    ("body", r"\bCao\s+M[eé]n\b", "Cao Mên", "body_cao_men"),
    ("body", r"\bcudp\b", "cướp", "body_cuop"),
    ("body", r"\bdem\b", "đem", "body_dem"),
    ("body", r"\bla\s+", "là ", "body_la"),
    ("body", r"\bthuoc\b", "thuộc", "body_thuoc"),
    ("body", r"\bthong\s+hạt\b", "thống hạt", "body_thong_hat"),
    ("body", r"\bki[eê]m\s+li\b", "kiêm lí", "body_kiem_li"),
    ("body", r"\bki[eê]m\s+lý\b", "kiêm lý", "body_kiem_ly"),
    ("body", r"\btrấn\s+th[uu]\b", "trấn thủ", "body_tran_thu"),
    ("body", r"\bb[óa]\s+ch[íi]nh\b", "bố chính", "body_bo_chinh"),
    ("body", r"\ban\s+s[áa]t\b", "án sát", "body_an_sat"),
    ("body", r"\bngach\s+thu[eế]\b", "ngạch thuế", "body_ngach_thue"),
    ("body", r"\b2\.192lang\b", "2.192 lạng", "body_2192_lang"),
    ("body", r"\bl[àa]bình\b", "là bình", "body_glued_la_binh"),
    ("body", r"\bđặthuyện\b", "đặt huyện", "body_glued_dat_huyen"),
    ("body", r"\bgặttháng\b", "gặt tháng", "body_glued_gat_thang"),
]

KNOWN_BAD_SUGGESTIONS: Dict[str, str] = {
    "QUOC": "QUỐC",
    "TRIEU": "TRIỀU",
    "NGUYEN": "NGUYỄN",
    "DAI": "ĐẠI",
    "NHAT": "NHẤT",
    "THONG": "THỐNG",
    "QUYEN": "QUYỂN",
    "TINH": "TỈNH",
    "HUYEN": "HUYỆN",
    "PHU": "PHỦ",
    "PHAN DA": "PHẦN DÃ",
    "DUNG DAT VA DIEN CACH": "DỰNG ĐẶT VÀ DIÊN CÁCH",
    "DUNG PAT VA DIEN CACH": "DỰNG ĐẶT VÀ DIÊN CÁCH",
    "HINH THE": "HÌNH THẾ",
    "KHI HAU": "KHÍ HẬU",
    "PHONG TUC": "PHONG TỤC",
    "THANH TRI": "THÀNH TRÌ",
    "TRUONG HOC": "TRƯỜNG HỌC",
    "dia giới": "địa giới",
    "phia": "phía",
    "dim": "dặm",
    "dam": "dặm",
    "cudp": "cướp",
    "bi€n": "biển",
    "ñăm": "năm",
    "Nim": "Năm",
}

# A small allow-list of common ASCII proper nouns / historical terms. This keeps
# suspicious scoring useful without flagging every no-accent token.
ASCII_ALLOW = {
    "Gia", "Long", "Minh", "Menh", "Mệnh", "Tu", "Tự", "Duc", "Đức", "Thieu", "Thiệu", "Tri", "Trị",
    "Le", "Lê", "Ly", "Lý", "Tran", "Trần", "Nguyen", "Nguyễn", "Quang", "Binh", "Dinh", "Hoa", "Hóa",
    "An", "Nam", "Bac", "Bắc", "Dong", "Đông", "Tay", "Tây", "Son", "Sơn", "Hai", "Hải", "Ha", "Hà",
    "Tien", "Tiên", "Giang", "Chau", "Châu", "Mau", "Cà", "Cao", "Men", "Mên", "Viet", "Việt",
}


# -----------------------------------------------------------------------------
# Data classes
# -----------------------------------------------------------------------------

@dataclass
class BlockRecord:
    work_id: str
    pdf_file: str
    page: int
    block_id: int
    bbox: List[float]
    raw_text: str
    clean_text: str
    type: str = "block"
    crop_path: str = ""


@dataclass
class ParagraphRecord:
    work_id: str
    pdf_file: str
    page: int
    para_id: int
    block_ids: List[int]
    bbox: List[float]
    raw_text: str
    clean_text: str
    auto_text: str
    type: str
    correction_rules: List[str] = field(default_factory=list)
    crop_path: str = ""


@dataclass
class SentenceRecord:
    work_id: str
    pdf_file: str
    page: int
    para_id: int
    sent_id: int
    block_ids: List[int]
    bbox: List[float]
    type: str
    raw_text: str
    clean_text: str
    auto_text: str
    final_text: str
    correction_rules: List[str] = field(default_factory=list)
    quality_score: int = 0
    reasons: List[str] = field(default_factory=list)
    suggestions: List[str] = field(default_factory=list)
    crop_path: str = ""
    manual_text: str = ""
    review_status: str = ""


# -----------------------------------------------------------------------------
# Basic helpers
# -----------------------------------------------------------------------------

def ensure_dir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path


def norm_unicode(text: str) -> str:
    if not text:
        return ""
    text = unicodedata.normalize("NFKC", text)
    replacements = {
        "\u00a0": " ",
        "\u200b": "",
        "\u200c": "",
        "\u200d": "",
        "\ufeff": "",
        "\r": "\n",
        "‐": "-",
        "‑": "-",
        "‒": "-",
        "–": "-",
        "—": "-",
        "−": "-",
        "…": "…",
        "´": "'",
        "`": "'",
        "“": "“",
        "”": "”",
        "‘": "'",
        "’": "'",
    }
    for k, v in replacements.items():
        text = text.replace(k, v)
    return text


def clean_spaces(text: str) -> str:
    text = norm_unicode(text)
    text = text.replace("\n", " ")
    text = MULTISPACE_RE.sub(" ", text)
    text = SPACE_BEFORE_PUNCT_RE.sub(r"\1", text)
    # Add space after comma/semicolon/colon/question/exclamation when glued to a letter.
    text = NO_SPACE_AFTER_PUNCT_RE.sub(r"\1 \2", text)
    # Do not globally split word.word because tr.C.N. and abbreviations exist.
    text = text.strip()
    return text


def text_hash(text: str, n: int = 10) -> str:
    return hashlib.sha1(text.encode("utf-8", errors="ignore")).hexdigest()[:n]


def union_bbox(boxes: Sequence[Sequence[float]]) -> List[float]:
    valid = [b for b in boxes if b and len(b) == 4]
    if not valid:
        return [0.0, 0.0, 0.0, 0.0]
    return [
        float(min(b[0] for b in valid)),
        float(min(b[1] for b in valid)),
        float(max(b[2] for b in valid)),
        float(max(b[3] for b in valid)),
    ]


def is_page_number_line(text: str) -> bool:
    s = clean_spaces(text)
    return bool(PAGE_NUMBER_RE.match(s) or ROMAN_PAGE_RE.match(s))


def is_pdf_noise_line(text: str) -> bool:
    s = clean_spaces(text)
    if not s:
        return True
    if PDF_FILENAME_RE.search(s):
        return True
    # Lines that are only punctuation/noise.
    if len(s) <= 4 and not re.search(r"[A-Za-zÀ-ỹĐđ]", s):
        return True
    return False


def strip_outer_noise(text: str) -> str:
    s = norm_unicode(text)
    s = re.sub(r"[\u0000-\u001F]+", " ", s)
    s = PDF_FILENAME_RE.sub("", s)
    s = clean_spaces(s)
    return s


def token_has_vietnamese_accent(tok: str) -> bool:
    # Check any char outside ASCII letters or Vietnamese đ.
    if "đ" in tok.lower():
        return True
    return any((ord(ch) > 127 and ch.isalpha()) for ch in tok)


def asciiless_ratio(text: str) -> float:
    toks = [t for t in TOKEN_RE.findall(text) if re.search(r"[A-Za-zÀ-ỹĐđ]", t)]
    if not toks:
        return 0.0
    no_acc = 0
    for t in toks:
        if t in ASCII_ALLOW:
            continue
        if re.fullmatch(r"[A-Za-z]+", t):
            no_acc += 1
    return no_acc / max(1, len(toks))


def looks_like_heading(text: str) -> bool:
    s = clean_spaces(text).strip(" .:-")
    if not s:
        return False
    up = strip_accents(s).upper()
    letters = [c for c in s if c.isalpha()]
    upper_ratio = 0.0
    if letters:
        upper_ratio = sum(1 for c in letters if c.upper() == c) / max(1, len(letters))

    if up in KNOWN_SECTION_HEADINGS:
        return True
    if re.match(r"^(DAI|ĐẠI)\s+NAM\s+", up) and upper_ratio > 0.60:
        return True
    if re.match(r"^QUYEN\s+([IVXLCDM]+|\d+)$", up) and upper_ratio > 0.60:
        return True
    # Administrative headings such as TINH HA TIEN are usually short, all/mostly caps,
    # and do not contain commas or long descriptive text. Do not classify body lines
    # like "tính An Giang 35 dặm, ..." as headings.
    if re.match(r"^(TINH|TỈNH|PHU|PHỦ|HUYEN|HUYỆN|CHAU|CHÂU)\s+", up):
        if len(s) <= 80 and upper_ratio > 0.60 and not re.search(r"[,;]", s):
            return True
    # Uppercase short line with no final punctuation is likely heading.
    if letters and len(s) <= 60 and UPPER_HEADING_RE.match(s):
        if upper_ratio > 0.8:
            return True
    return False


def strip_accents(text: str) -> str:
    # Keep Đ/đ mapped to D/d.
    text = text.replace("Đ", "D").replace("đ", "d")
    decomposed = unicodedata.normalize("NFD", text)
    return "".join(ch for ch in decomposed if unicodedata.category(ch) != "Mn")


def normalize_heading(text: str) -> Tuple[str, List[str]]:
    s = clean_spaces(text).strip(" .")
    key = strip_accents(s).upper()
    key = re.sub(r"\s+", " ", key)
    if key in KNOWN_SECTION_HEADINGS:
        return KNOWN_SECTION_HEADINGS[key], ["heading_dictionary"]
    return s, []


# -----------------------------------------------------------------------------
# Correction rules
# -----------------------------------------------------------------------------

def load_custom_rules(path: Optional[Path]) -> List[Tuple[str, str, str, str]]:
    """Load CSV rules: scope,pattern,replacement,rule_name.

    scope can be all, heading, body. Header row is required.
    """
    if not path:
        return []
    if not path.exists():
        raise FileNotFoundError(f"Custom rule file not found: {path}")
    rules: List[Tuple[str, str, str, str]] = []
    with path.open("r", encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)
        required = {"pattern", "replacement"}
        if not required.issubset(set(reader.fieldnames or [])):
            raise ValueError("Custom rule CSV must include columns: pattern,replacement. Optional: scope,rule_name")
        for i, row in enumerate(reader, start=2):
            pattern = (row.get("pattern") or "").strip()
            repl = row.get("replacement") or ""
            if not pattern:
                continue
            scope = (row.get("scope") or "all").strip().lower()
            if scope not in {"all", "heading", "body"}:
                raise ValueError(f"Invalid scope at row {i}: {scope}")
            name = (row.get("rule_name") or f"custom_rule_{i}").strip()
            rules.append((scope, pattern, repl, name))
    return rules


def apply_rules(text: str, rec_type: str, custom_rules: Sequence[Tuple[str, str, str, str]] = ()) -> Tuple[str, List[str]]:
    """Apply conservative regex corrections and return changed text + rule names."""
    s = clean_spaces(text)
    rules_applied: List[str] = []

    if rec_type == "heading":
        normalized, heading_rules = normalize_heading(s)
        if normalized != s:
            s = normalized
            rules_applied.extend(heading_rules)

    for scope, pattern, repl, name in list(BUILTIN_RULES) + list(custom_rules):
        if scope != "all" and scope != rec_type:
            continue
        new_s = re.sub(pattern, repl, s)
        if new_s != s:
            s = new_s
            rules_applied.append(name)

    s = clean_spaces(s)
    return s, rules_applied


# -----------------------------------------------------------------------------
# PDF extraction
# -----------------------------------------------------------------------------

def reconstruct_block_text(block: Dict[str, Any]) -> str:
    lines_out: List[str] = []
    for line in block.get("lines", []):
        spans = line.get("spans", [])
        spans = sorted(spans, key=lambda sp: sp.get("bbox", [0, 0, 0, 0])[0])
        txt = "".join(sp.get("text", "") for sp in spans)
        txt = strip_outer_noise(txt)
        if txt:
            lines_out.append(txt)
    return "\n".join(lines_out)


In [4]:
def extract_pdf_blocks(pdf_path: Path, work_id: str, start_page: int = 1, end_page: Optional[int] = None) -> List[BlockRecord]:
    doc = fitz.open(str(pdf_path))
    total_pages = doc.page_count
    if end_page is None:
        end_page = total_pages
    start_page = max(1, start_page)
    end_page = min(total_pages, end_page)
    records: List[BlockRecord] = []

    for page_num in tqdm(range(start_page, end_page + 1), desc=f"extract blocks {pdf_path.name}"):
        page = doc.load_page(page_num - 1)
        data = page.get_text("dict")
        blocks = [b for b in data.get("blocks", []) if b.get("type", 0) == 0]
        blocks = sorted(blocks, key=lambda b: (round(b.get("bbox", [0, 0, 0, 0])[1], 1), round(b.get("bbox", [0, 0, 0, 0])[0], 1)))
        for local_id, block in enumerate(blocks):
            raw = reconstruct_block_text(block)
            if not raw:
                continue
            # Drop obvious footer/page number/file noise at line level.
            kept_lines = []
            for line in raw.splitlines():
                line_clean = strip_outer_noise(line)
                if is_pdf_noise_line(line_clean):
                    continue
                if is_page_number_line(line_clean):
                    continue
                kept_lines.append(line_clean)
            raw_kept = "\n".join(kept_lines).strip()
            if not raw_kept:
                continue
            bbox = [float(x) for x in block.get("bbox", [0, 0, 0, 0])]
            records.append(BlockRecord(
                work_id=work_id,
                pdf_file=pdf_path.name,
                page=page_num,
                block_id=local_id,
                bbox=bbox,
                raw_text=raw_kept,
                clean_text=clean_spaces(raw_kept),
            ))
    doc.close()
    return records


def should_join_lines(prev: str, cur: str) -> bool:
    prev_s = clean_spaces(prev)
    cur_s = clean_spaces(cur)
    if not prev_s or not cur_s:
        return False
    if looks_like_heading(prev_s) or looks_like_heading(cur_s):
        return False
    if FOOTNOTE_START_RE.match(cur_s):
        return False
    if LIST_ITEM_RE.match(cur_s):
        return False
    # If previous line ends with hyphen and next starts lowercase, join without extra space later.
    if prev_s.endswith("-"):
        return True
    # Colon often starts explanation. Keep same paragraph if current starts lowercase.
    if HEADING_TERMINAL_RE.search(prev_s) and re.match(r"^[a-zà-ỹđ]", cur_s):
        return True
    if TERMINAL_RE.search(prev_s):
        return False
    # If next starts lowercase/digit, usually continuation.
    if re.match(r"^[a-zà-ỹđ0-9]", cur_s):
        return True
    # If previous has comma/semicolon, usually continuation.
    if re.search(r"[,;]$", prev_s):
        return True
    # Long historical paragraphs often continue with uppercase proper names. Join if prev clearly not closed.
    return True


def reflow_lines_to_paragraphs(lines: List[str]) -> List[str]:
    paras: List[str] = []
    current = ""
    for raw_line in lines:
        line = strip_outer_noise(raw_line)
        if not line or is_page_number_line(line) or is_pdf_noise_line(line):
            continue
        if not current:
            current = line
            continue
        if should_join_lines(current, line):
            if current.endswith("-"):
                current = current[:-1] + line
            else:
                current = current + " " + line
        else:
            paras.append(clean_spaces(current))
            current = line
    if current:
        paras.append(clean_spaces(current))
    return paras


def blocks_to_paragraphs(
    blocks: Sequence[BlockRecord],
    enable_auto_correction: bool = True,
    custom_rules: Sequence[Tuple[str, str, str, str]] = (),
) -> List[ParagraphRecord]:
    """Convert PDF text blocks to page-level paragraphs.

    Many scanned PDFs expose each printed line as a separate PyMuPDF block. If we
    reflow only inside each block, sentences become chopped line-by-line. This
    function therefore reflows across all blocks on a page while keeping block ids
    and a union bbox for provenance/crops.
    """
    records: List[ParagraphRecord] = []
    by_page: Dict[Tuple[str, str, int], List[BlockRecord]] = {}
    for b in blocks:
        by_page.setdefault((b.work_id, b.pdf_file, b.page), []).append(b)

    for (work_id, pdf_file, page), page_blocks in sorted(by_page.items(), key=lambda x: x[0]):
        page_blocks = sorted(page_blocks, key=lambda b: (b.bbox[1], b.bbox[0]))

        # Flatten blocks to ordered line entries. Most pages are single-column;
        # sorting by y then x is enough for this corpus.
        line_entries: List[Tuple[str, int, List[float]]] = []
        for block in page_blocks:
            for line in block.raw_text.splitlines():
                line = strip_outer_noise(line)
                if not line or is_pdf_noise_line(line) or is_page_number_line(line):
                    continue
                line_entries.append((line, block.block_id, block.bbox))

        para_idx = 0
        cur_text = ""
        cur_ids: List[int] = []
        cur_boxes: List[List[float]] = []

        def flush_current() -> None:
            nonlocal para_idx, cur_text, cur_ids, cur_boxes
            para = clean_spaces(cur_text)
            if not para:
                cur_text, cur_ids, cur_boxes = "", [], []
                return
            rec_type = "heading" if looks_like_heading(para) else "footnote" if FOOTNOTE_START_RE.match(para) else "paragraph"
            auto = para
            rules: List[str] = []
            if enable_auto_correction:
                auto, rules = apply_rules(para, "heading" if rec_type == "heading" else "body", custom_rules=custom_rules)
            records.append(ParagraphRecord(
                work_id=work_id,
                pdf_file=pdf_file,
                page=page,
                para_id=para_idx,
                block_ids=sorted(set(cur_ids)),
                bbox=union_bbox(cur_boxes),
                raw_text=para,
                clean_text=clean_spaces(para),
                auto_text=auto,
                type=rec_type,
                correction_rules=rules,
            ))
            para_idx += 1
            cur_text, cur_ids, cur_boxes = "", [], []

        for line, block_id, bbox in line_entries:
            line_is_heading = looks_like_heading(line)
            if not cur_text:
                cur_text = line
                cur_ids = [block_id]
                cur_boxes = [bbox]
                # Standalone headings should become their own paragraph immediately.
                if line_is_heading:
                    flush_current()
                continue

            # A new heading always starts a new paragraph.
            if line_is_heading:
                flush_current()
                cur_text = line
                cur_ids = [block_id]
                cur_boxes = [bbox]
                flush_current()
                continue

            # Footnotes start a new paragraph so they do not pollute body text.
            if FOOTNOTE_START_RE.match(line):
                flush_current()
                cur_text = line
                cur_ids = [block_id]
                cur_boxes = [bbox]
                continue

            if should_join_lines(cur_text, line):
                if cur_text.endswith("-"):
                    cur_text = cur_text[:-1] + line
                else:
                    cur_text = cur_text + " " + line
                cur_ids.append(block_id)
                cur_boxes.append(bbox)
            else:
                flush_current()
                cur_text = line
                cur_ids = [block_id]
                cur_boxes = [bbox]

        flush_current()
    return records


# -----------------------------------------------------------------------------
# Sentence segmentation
# -----------------------------------------------------------------------------

def protect_abbreviations(text: str) -> Tuple[str, Dict[str, str]]:
    mapping: Dict[str, str] = {}
    s = text
    for i, abbr in enumerate(ABBREVIATIONS):
        if abbr in s:
            key = f"<ABBR{i}>"
            s = s.replace(abbr, key)
            mapping[key] = abbr
    return s, mapping


def restore_abbreviations(text: str, mapping: Dict[str, str]) -> str:
    s = text
    for key, val in mapping.items():
        s = s.replace(key, val)
    return s


def regex_sentence_split(text: str) -> List[str]:
    s = clean_spaces(text)
    if not s:
        return []
    protected, mapping = protect_abbreviations(s)
    # Split at terminal punctuation when followed by quote/paren optional + space + uppercase/accent digit quote.
    pieces: List[str] = []
    start = 0
    for m in re.finditer(r"[\.\?!…]+[\)\]\}”’\"]?\s+(?=[A-ZÀ-ỸĐ\(\"“])", protected):
        end = m.end()
        pieces.append(protected[start:end].strip())
        start = end
    tail = protected[start:].strip()
    if tail:
        pieces.append(tail)
    pieces = [restore_abbreviations(p, mapping).strip() for p in pieces if p.strip()]
    # Merge very short dangling pieces.
    merged: List[str] = []
    for p in pieces:
        if merged and len(p) < 12 and not TERMINAL_RE.search(merged[-1]):
            merged[-1] = clean_spaces(merged[-1] + " " + p)
        else:
            merged.append(p)
    return merged


def split_sentences(text: str, rec_type: str, use_underthesea: bool = True) -> List[str]:
    s = clean_spaces(text)
    if not s:
        return []
    if rec_type in {"heading"}:
        return [s]
    # Footnotes are often one record even if they contain many punctuation marks.
    if rec_type == "footnote" and len(s) < 300:
        return [s]
    if use_underthesea and underthesea_sent_tokenize is not None:
        try:
            out = [clean_spaces(x) for x in underthesea_sent_tokenize(s) if clean_spaces(x)]
            if out:
                return out
        except Exception:
            pass
    return regex_sentence_split(s)


def paragraphs_to_sentences(paragraphs: Sequence[ParagraphRecord], use_underthesea: bool = True) -> List[SentenceRecord]:
    records: List[SentenceRecord] = []
    sent_global: Dict[str, int] = {}
    for p in paragraphs:
        key = p.work_id
        sent_global.setdefault(key, 0)
        sentence_texts = split_sentences(p.auto_text, p.type, use_underthesea=use_underthesea)
        raw_sentence_texts = split_sentences(p.raw_text, p.type, use_underthesea=False)
        for local_idx, auto_sent in enumerate(sentence_texts):
            raw_sent = raw_sentence_texts[local_idx] if local_idx < len(raw_sentence_texts) else p.raw_text
            sent_id = sent_global[key]
            sent_global[key] += 1
            qscore, reasons, suggestions = score_suspicious(auto_sent, raw_sent, p.type, p.correction_rules)
            records.append(SentenceRecord(
                work_id=p.work_id,
                pdf_file=p.pdf_file,
                page=p.page,
                para_id=p.para_id,
                sent_id=sent_id,
                block_ids=p.block_ids,
                bbox=p.bbox,
                type=p.type if p.type == "heading" else "sentence" if p.type == "paragraph" else p.type,
                raw_text=raw_sent,
                clean_text=clean_spaces(raw_sent),
                auto_text=auto_sent,
                final_text=auto_sent,
                correction_rules=list(p.correction_rules),
                quality_score=qscore,
                reasons=reasons,
                suggestions=suggestions,
                crop_path=p.crop_path,
            ))
    return records


In [5]:
def score_suspicious(text: str, raw_text: str, rec_type: str, correction_rules: Sequence[str]) -> Tuple[int, List[str], List[str]]:
    reasons: List[str] = []
    suggestions: List[str] = []
    score = 0
    s = text or ""
    raw = raw_text or ""

    junk = JUNK_CHARS_RE.findall(raw + " " + s)
    if junk:
        score += min(5, len(set(junk)) + 1)
        reasons.append("junk_chars:" + "".join(sorted(set(junk))))

    if not EXPECTED_TEXT_CHARS_RE.match(s):
        score += 2
        reasons.append("unexpected_unicode")

    if GLUED_NUM_LETTER_RE.search(s):
        score += 2
        reasons.append("glued_number_letter")

    no_acc_ratio = asciiless_ratio(s)
    if rec_type != "heading" and no_acc_ratio >= 0.35:
        score += 3
        reasons.append(f"high_no_accent_ratio:{no_acc_ratio:.2f}")
    elif rec_type == "heading" and no_acc_ratio >= 0.25:
        score += 2
        reasons.append(f"heading_no_accent_ratio:{no_acc_ratio:.2f}")

    if re.search(r"\b[A-Z]{3,}\b", s) and rec_type != "heading":
        score += 1
        reasons.append("uppercase_token_in_body")

    if re.search(r"[;:,]\S", raw):
        score += 1
        reasons.append("missing_space_after_punct_raw")

    if len(s) > 350:
        score += 2
        reasons.append("very_long_sentence")
    if len(s) < 5 and rec_type != "heading":
        score += 1
        reasons.append("very_short_sentence")

    # Known bad substrings. Match ASCII bad tokens literally, not accent-stripped,
    # otherwise TINH would also match the valid word "tình".
    raw_low = raw.lower()
    s_low = s.lower()
    combined = raw + "\n" + s
    for bad, sug in KNOWN_BAD_SUGGESTIONS.items():
        if bad.isascii():
            if " " in bad:
                hit = re.search(re.escape(bad), combined, flags=re.I) is not None
            else:
                hit = re.search(r"\b" + re.escape(bad) + r"\b", combined, flags=re.I) is not None
        else:
            bad_low = bad.lower()
            hit = bad_low in raw_low or bad_low in s_low
        if hit:
            if sug not in suggestions:
                suggestions.append(f"{bad} -> {sug}")
            score += 1
    if correction_rules:
        score += 1
        reasons.append("auto_rules_applied")

    # Flag if raw and auto differ substantially.
    if raw and s and raw != s:
        dist_ratio = rough_change_ratio(raw, s)
        if dist_ratio > 0.25:
            score += 2
            reasons.append(f"large_auto_change:{dist_ratio:.2f}")

    # Cap score for easier sorting.
    return min(score, 20), reasons, suggestions[:8]


def rough_change_ratio(a: str, b: str) -> float:
    # Fast approximation without external Levenshtein.
    a_tokens = TOKEN_RE.findall(a)
    b_tokens = TOKEN_RE.findall(b)
    if not a_tokens and not b_tokens:
        return 0.0
    aset = set(a_tokens)
    bset = set(b_tokens)
    overlap = len(aset & bset)
    denom = max(1, max(len(aset), len(bset)))
    return 1.0 - overlap / denom


def collect_spell_candidates(sentences: Sequence[SentenceRecord], min_count: int = 2) -> pd.DataFrame:
    counts: Dict[str, Dict[str, Any]] = {}
    suspicious_token_re = re.compile(r"^[A-Za-z]{3,}$|.*[€©®ˆ￾|_].*|.*\d+[A-Za-zÀ-ỹĐđ]+.*|.*[A-Za-zÀ-ỹĐđ]+\d+.*")
    for r in sentences:
        for tok in TOKEN_RE.findall(r.raw_text + " " + r.auto_text):
            if tok in ASCII_ALLOW:
                continue
            bad = False
            if suspicious_token_re.match(tok):
                bad = True
            if len(tok) >= 4 and re.fullmatch(r"[A-Za-z]+", tok) and tok not in ASCII_ALLOW:
                bad = True
            if JUNK_CHARS_RE.search(tok):
                bad = True
            if bad:
                item = counts.setdefault(tok, {"token": tok, "count": 0, "examples": []})
                item["count"] += 1
                if len(item["examples"]) < 3:
                    item["examples"].append(f"{r.work_id}:p{r.page}: {r.raw_text[:180]}")
    rows = []
    for item in counts.values():
        if item["count"] >= min_count:
            rows.append({
                "token": item["token"],
                "count": item["count"],
                "suggestion": KNOWN_BAD_SUGGESTIONS.get(item["token"], ""),
                "examples": " || ".join(item["examples"]),
            })
    rows.sort(key=lambda x: (-x["count"], x["token"]))
    return pd.DataFrame(rows)


# -----------------------------------------------------------------------------
# Rendering/crops
# -----------------------------------------------------------------------------

def render_page_crop(
    pdf_path: Path,
    page_num: int,
    bbox: Sequence[float],
    out_path: Path,
    dpi: int = 180,
    padding_px: int = 10,
) -> Optional[str]:
    if Image is None:
        return None
    try:
        doc = fitz.open(str(pdf_path))
        page = doc.load_page(page_num - 1)
        zoom = dpi / 72.0
        mat = fitz.Matrix(zoom, zoom)
        pix = page.get_pixmap(matrix=mat, alpha=False)
        img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
        x0, y0, x1, y1 = bbox
        crop_box = (
            max(0, int(x0 * zoom) - padding_px),
            max(0, int(y0 * zoom) - padding_px),
            min(img.width, int(x1 * zoom) + padding_px),
            min(img.height, int(y1 * zoom) + padding_px),
        )
        if crop_box[2] <= crop_box[0] or crop_box[3] <= crop_box[1]:
            return None
        crop = img.crop(crop_box)
        ensure_dir(out_path.parent)
        crop.save(out_path)
        doc.close()
        return str(out_path)
    except Exception:
        return None


def add_crops_to_sentences(
    sentences: List[SentenceRecord],
    pdf_lookup: Dict[str, Path],
    out_dir: Path,
    mode: str = "suspicious",
    dpi: int = 180,
) -> None:
    if mode == "none":
        return
    crop_dir = ensure_dir(out_dir / "crops")
    cache: Dict[Tuple[str, int, Tuple[int, int, int, int]], str] = {}
    targets = sentences if mode == "all" else [r for r in sentences if r.quality_score > 0]
    for r in tqdm(targets, desc="render crops"):
        pdf_path = pdf_lookup.get(r.pdf_file)
        if not pdf_path:
            continue
        bbox_key = tuple(int(round(x)) for x in r.bbox)
        key = (r.pdf_file, r.page, bbox_key)
        if key in cache:
            r.crop_path = cache[key]
            continue
        name = f"{Path(r.pdf_file).stem}_p{r.page:04d}_b{'-'.join(map(str, r.block_ids))}_{text_hash(r.raw_text)}.png"
        out_path = crop_dir / Path(r.pdf_file).stem / name
        saved = render_page_crop(pdf_path, r.page, r.bbox, out_path, dpi=dpi)
        if saved:
            rel = os.path.relpath(saved, out_dir)
            cache[key] = rel
            r.crop_path = rel


# -----------------------------------------------------------------------------
# IO and reports
# -----------------------------------------------------------------------------

def write_jsonl(path: Path, records: Iterable[Any]) -> None:
    ensure_dir(path.parent)
    with path.open("w", encoding="utf-8") as f:
        for r in records:
            d = asdict(r) if hasattr(r, "__dataclass_fields__") else dict(r)
            f.write(json.dumps(d, ensure_ascii=False) + "\n")


def read_jsonl(path: Path) -> List[Dict[str, Any]]:
    rows: List[Dict[str, Any]] = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def sentences_to_dataframe(sentences: Sequence[SentenceRecord]) -> pd.DataFrame:
    rows = []
    for r in sentences:
        rows.append({
            "work_id": r.work_id,
            "pdf_file": r.pdf_file,
            "page": r.page,
            "para_id": r.para_id,
            "sent_id": r.sent_id,
            "type": r.type,
            "quality_score": r.quality_score,
            "reasons": "; ".join(r.reasons),
            "suggestions": " | ".join(r.suggestions),
            "correction_rules": "; ".join(r.correction_rules),
            "raw_text": r.raw_text,
            "clean_text": r.clean_text,
            "auto_text": r.auto_text,
            "corrected_text_manual": r.manual_text,
            "review_status": r.review_status,
            "final_text": r.final_text,
            "crop_path": r.crop_path,
            "bbox": json.dumps(r.bbox, ensure_ascii=False),
            "block_ids": json.dumps(r.block_ids, ensure_ascii=False),
        })
    return pd.DataFrame(rows)


def export_text_files(sentences: Sequence[SentenceRecord], out_dir: Path, field: str = "final_text", prefix: str = "final") -> None:
    by_work: Dict[str, List[SentenceRecord]] = {}
    for r in sentences:
        by_work.setdefault(r.work_id, []).append(r)
    text_dir = ensure_dir(out_dir / "texts")
    all_lines: List[str] = []
    for work_id, rows in sorted(by_work.items()):
        rows = sorted(rows, key=lambda r: (r.page, r.sent_id))
        path = text_dir / f"{work_id}_{prefix}.txt"
        with path.open("w", encoding="utf-8") as f:
            last_page = None
            for r in rows:
                if last_page != r.page:
                    f.write(f"\n\n### {work_id} - page {r.page}\n")
                    last_page = r.page
                text = getattr(r, field)
                f.write(text.strip() + "\n")
                all_lines.append(f"{work_id}\tp{r.page}\t{r.sent_id}\t{text.strip()}")
    with (text_dir / f"all_{prefix}.tsv").open("w", encoding="utf-8") as f:
        f.write("work_id\tpage\tsent_id\ttext\n")
        for line in all_lines:
            f.write(line + "\n")


def make_review_dataframe(df_all: pd.DataFrame, review_mode: str, min_score: int = 1) -> pd.DataFrame:
    if review_mode == "all":
        df = df_all.copy()
    elif review_mode == "auto_rules":
        df = df_all[df_all["correction_rules"].fillna("").astype(str).str.len() > 0].copy()
    else:
        df = df_all[df_all["quality_score"].fillna(0).astype(int) >= min_score].copy()
    # Recommended manual edit starts with auto_text. Reviewer changes only when needed.
    if "corrected_text_manual" in df.columns:
        df["corrected_text_manual"] = df["auto_text"]
    if "review_status" in df.columns:
        df["review_status"] = ""
    sort_cols = ["quality_score", "work_id", "page", "sent_id"]
    df = df.sort_values(sort_cols, ascending=[False, True, True, True])
    return df


def write_excel(path: Path, sheets: Dict[str, pd.DataFrame]) -> None:
    ensure_dir(path.parent)
    with pd.ExcelWriter(path, engine="openpyxl") as writer:
        for sheet_name, df in sheets.items():
            safe_name = sheet_name[:31]
            df.to_excel(writer, sheet_name=safe_name, index=False)
        wb = writer.book
        for ws in wb.worksheets:
            ws.freeze_panes = "A2"
            ws.auto_filter.ref = ws.dimensions
            for col in ws.columns:
                header = col[0].value
                letter = col[0].column_letter
                if header in {"raw_text", "clean_text", "auto_text", "corrected_text_manual", "final_text", "reasons", "suggestions"}:
                    ws.column_dimensions[letter].width = 60
                    for cell in col:
                        cell.alignment = cell.alignment.copy(wrap_text=True, vertical="top")
                elif header in {"crop_path", "correction_rules", "bbox", "block_ids"}:
                    ws.column_dimensions[letter].width = 32
                    for cell in col:
                        cell.alignment = cell.alignment.copy(wrap_text=True, vertical="top")
                else:
                    ws.column_dimensions[letter].width = min(24, max(10, len(str(header or "")) + 2))
            # Make crop paths clickable when possible.
            header_map = {cell.value: idx + 1 for idx, cell in enumerate(ws[1])}
            if "crop_path" in header_map:
                cidx = header_map["crop_path"]
                for row in range(2, ws.max_row + 1):
                    cell = ws.cell(row=row, column=cidx)
                    if cell.value:
                        cell.hyperlink = str(cell.value)
                        cell.style = "Hyperlink"


def build_summary(blocks: Sequence[BlockRecord], paras: Sequence[ParagraphRecord], sents: Sequence[SentenceRecord]) -> pd.DataFrame:
    rows = []
    by_work: Dict[str, List[SentenceRecord]] = {}
    for r in sents:
        by_work.setdefault(r.work_id, []).append(r)
    block_counts: Dict[str, int] = {}
    para_counts: Dict[str, int] = {}
    for b in blocks:
        block_counts[b.work_id] = block_counts.get(b.work_id, 0) + 1
    for p in paras:
        para_counts[p.work_id] = para_counts.get(p.work_id, 0) + 1
    for work_id, records in sorted(by_work.items()):
        scores = [r.quality_score for r in records]
        rows.append({
            "work_id": work_id,
            "pages": len(set(r.page for r in records)),
            "blocks": block_counts.get(work_id, 0),
            "paragraphs": para_counts.get(work_id, 0),
            "sentences": len(records),
            "suspicious_sentences": sum(1 for r in records if r.quality_score > 0),
            "high_suspicious_score_ge_5": sum(1 for r in records if r.quality_score >= 5),
            "auto_rule_sentences": sum(1 for r in records if r.correction_rules),
            "avg_quality_score": round(statistics.mean(scores), 3) if scores else 0,
            "max_quality_score": max(scores) if scores else 0,
        })
    return pd.DataFrame(rows)


def save_outputs(
    out_dir: Path,
    blocks: Sequence[BlockRecord],
    paras: Sequence[ParagraphRecord],
    sents: Sequence[SentenceRecord],
    review_mode: str = "suspicious",
    review_min_score: int = 1,
) -> None:
    ensure_dir(out_dir)
    data_dir = ensure_dir(out_dir / "data")
    review_dir = ensure_dir(out_dir / "review")

    write_jsonl(data_dir / "blocks.jsonl", blocks)
    write_jsonl(data_dir / "paragraphs.jsonl", paras)
    write_jsonl(data_dir / "sentences.jsonl", sents)

    df_all = sentences_to_dataframe(sents)
    df_all.to_csv(data_dir / "sentences.csv", index=False, encoding="utf-8-sig")
    df_summary = build_summary(blocks, paras, sents)
    df_summary.to_csv(out_dir / "summary.csv", index=False, encoding="utf-8-sig")

    df_review = make_review_dataframe(df_all, review_mode=review_mode, min_score=review_min_score)
    df_candidates = collect_spell_candidates(sents)
    df_candidates.to_csv(review_dir / "spell_candidates.csv", index=False, encoding="utf-8-sig")
    df_review.to_csv(review_dir / "suspicious_review.csv", index=False, encoding="utf-8-sig")

    write_excel(
        review_dir / "suspicious_review.xlsx",
        {
            "review": df_review,
            "spell_candidates": df_candidates,
            "summary": df_summary,
        },
    )
    export_text_files(sents, out_dir, field="final_text", prefix="auto")


# -----------------------------------------------------------------------------
# Manual correction apply
# -----------------------------------------------------------------------------

def load_review_table(review_path: Path) -> pd.DataFrame:
    if review_path.suffix.lower() in {".xlsx", ".xlsm", ".xls"}:
        return pd.read_excel(review_path, sheet_name="review")
    return pd.read_csv(review_path, encoding="utf-8-sig")


def normalize_nan(value: Any) -> str:
    if value is None:
        return ""
    if isinstance(value, float) and math.isnan(value):
        return ""
    return str(value)


def apply_manual_review(out_dir: Path, review_path: Path, prefer_manual_nonempty: bool = True) -> List[SentenceRecord]:
    data_path = out_dir / "data" / "sentences.jsonl"
    if not data_path.exists():
        raise FileNotFoundError(f"Missing extracted sentences: {data_path}. Run extract first.")
    base_rows = read_jsonl(data_path)
    records: List[SentenceRecord] = [SentenceRecord(**row) for row in base_rows]
    by_key: Dict[Tuple[str, int, int], SentenceRecord] = {(r.work_id, int(r.page), int(r.sent_id)): r for r in records}

    review_df = load_review_table(review_path)
    changed = 0
    rejected = 0
    for _, row in review_df.iterrows():
        work_id = normalize_nan(row.get("work_id"))
        if not work_id:
            continue
        try:
            page = int(row.get("page"))
            sent_id = int(row.get("sent_id"))
        except Exception:
            continue
        rec = by_key.get((work_id, page, sent_id))
        if not rec:
            continue
        status = normalize_nan(row.get("review_status")).strip().lower()
        manual = normalize_nan(row.get("corrected_text_manual")).strip()
        rec.review_status = status
        rec.manual_text = manual
        if status in {"skip", "ok", "accept_auto", "auto_ok"}:
            rec.final_text = rec.auto_text
        elif status in {"reject_auto", "use_raw", "raw"}:
            rec.final_text = rec.clean_text or rec.raw_text
            rejected += 1
        elif manual and prefer_manual_nonempty:
            rec.final_text = manual
            if manual != rec.auto_text:
                changed += 1
        else:
            rec.final_text = rec.auto_text

    final_dir = ensure_dir(out_dir / "final")
    write_jsonl(final_dir / "final_sentences.jsonl", records)
    df_final = sentences_to_dataframe(records)
    df_final.to_csv(final_dir / "final_sentences.csv", index=False, encoding="utf-8-sig")
    export_text_files(records, final_dir, field="final_text", prefix="final")
    report = pd.DataFrame([{
        "review_file": str(review_path),
        "total_sentences": len(records),
        "manual_changed": changed,
        "auto_rejected_to_raw": rejected,
    }])
    report.to_csv(final_dir / "apply_report.csv", index=False, encoding="utf-8-sig")
    return records


In [6]:
def write_default_custom_rules(path: Path) -> None:
    ensure_dir(path.parent)
    rows = [
        {"scope": "body", "pattern": r"\bSAI_OCR\b", "replacement": "ĐÚNG", "rule_name": "example_replace_sai_ocr"},
        {"scope": "heading", "pattern": r"\bTEN_MUC_SAI\b", "replacement": "TÊN MỤC ĐÚNG", "rule_name": "example_heading"},
    ]
    with path.open("w", encoding="utf-8-sig", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["scope", "pattern", "replacement", "rule_name"])
        writer.writeheader()
        writer.writerows(rows)


In [7]:

from pathlib import Path
import re
import pandas as pd

PDF_SEARCH_DIRS = [
    VIET_DATA_DIR,
    KAGGLE_INPUT,
    REPO_DIR,
    Path("/mnt/data"),
]

def _safe_strip_accents_for_name(s: str) -> str:
    # Dùng hàm strip_accents đã định nghĩa trong core nếu có.
    try:
        return strip_accents(s)
    except Exception:
        import unicodedata
        return "".join(c for c in unicodedata.normalize("NFD", s) if unicodedata.category(c) != "Mn")

def infer_work_id_from_pdf_name(path: Path) -> str:
    name = _safe_strip_accents_for_name(path.name.lower())
    stem = _safe_strip_accents_for_name(path.stem.lower())

    # Ưu tiên q1..q5.
    m = re.search(r"(^|[^0-9])q\s*([1-5])([^0-9]|$)", stem)
    if m:
        return f"q{m.group(2)}"

    # Tên kiểu T1, tap1, tập 1, -T3le, T4...
    patterns = [
        r"(^|[^0-9])t\s*([1-5])([^0-9]|$)",
        r"tap\s*([1-5])",
        r"tập\s*([1-5])",
    ]
    for pat in patterns:
        m = re.search(pat, name)
        if m:
            # Pattern đầu có 3 group, các pattern sau có 1 group.
            vol = m.group(2) if len(m.groups()) >= 2 and m.group(2) else m.group(1)
            return f"q{vol}"

    # Fallback theo tên file.
    return path.stem

def discover_5_pdfs(search_dirs):
    all_pdfs = []
    for d in search_dirs:
        if d and d.exists():
            all_pdfs.extend(sorted(d.rglob("*.pdf")))

    # Deduplicate theo absolute path.
    unique = {}
    for p in all_pdfs:
        unique[str(p.resolve())] = p
    all_pdfs = list(unique.values())

    by_work = {}
    for p in all_pdfs:
        wid = infer_work_id_from_pdf_name(p)
        if wid in {"q1", "q2", "q3", "q4", "q5"}:
            # Ưu tiên file tên q*.pdf nếu có.
            if wid not in by_work or p.name.lower() == f"{wid}.pdf":
                by_work[wid] = p

    missing = [f"q{i}" for i in range(1, 6) if f"q{i}" not in by_work]
    if missing:
        print("Chưa map đủ q1..q5:", missing)
        print("Các PDF tìm thấy:")
        for p in all_pdfs[:50]:
            print("-", p)
        raise FileNotFoundError("Không tìm đủ 5 PDF. Hãy sửa PDF_SEARCH_DIRS hoặc đặt tên file theo q1..q5/T1..T5.")

    pdf_files = [by_work[f"q{i}"] for i in range(1, 6)]
    return pdf_files

pdf_files = discover_5_pdfs(PDF_SEARCH_DIRS)

print("PDF files sẽ xử lý:")
for p in pdf_files:
    print(infer_work_id_from_pdf_name(p), "=>", p)


PDF files sẽ xử lý:
q1 => /kaggle/input/datasets/quachthanhhmd/sinonom-local/vietnam/q1.pdf
q2 => /kaggle/input/datasets/quachthanhhmd/sinonom-local/vietnam/q2.pdf
q3 => /kaggle/input/datasets/quachthanhhmd/sinonom-local/vietnam/q3.pdf
q4 => /kaggle/input/datasets/quachthanhhmd/sinonom-local/vietnam/q4.pdf
q5 => /kaggle/input/datasets/quachthanhhmd/sinonom-local/vietnam/q5.pdf


In [8]:

# Chọn mode:
# - "content": bỏ bìa/trang phụ, lấy từ nội dung chính.
# - "all": lấy toàn bộ trang.
PAGE_MODE = "content"

# Page range là 1-indexed, end_page=None nghĩa là đến hết file.
PAGE_RANGES_CONTENT = {
    "q1": (13, None),  # Tập 1: bắt đầu từ phần Kinh sư; đổi về (1, None) nếu muốn lấy lời nói đầu.
    "q2": (3, None),
    "q3": (3, None),
    "q4": (4, None),
    "q5": (3, None),
}

PAGE_RANGES_ALL = {
    "q1": (1, None),
    "q2": (1, None),
    "q3": (1, None),
    "q4": (1, None),
    "q5": (1, None),
}

PAGE_RANGES = PAGE_RANGES_CONTENT if PAGE_MODE == "content" else PAGE_RANGES_ALL

# Review mode:
# - suspicious: chỉ câu/heading nghi lỗi
# - all: tất cả sentence records
# - auto_rules: chỉ dòng đã có rule auto correction tác động
REVIEW_MODE = "suspicious"
REVIEW_MIN_SCORE = 1

# Crop ảnh scan để đối chiếu:
# - off: không render crop
# - suspicious: chỉ render dòng nghi lỗi
# - all: render toàn bộ, khá nặng
RENDER_CROPS = "suspicious"
CROP_DPI = 170

USE_UNDERTHESEA = True
ENABLE_AUTO_CORRECTION = True

CONFIG_DIR = OUTPUT_DIR / "configs"
CONFIG_DIR.mkdir(parents=True, exist_ok=True)
CUSTOM_RULES_PATH = CONFIG_DIR / "custom_ocr_rules.csv"

# Tạo file rule mẫu nếu chưa có.
if not CUSTOM_RULES_PATH.exists():
    write_default_custom_rules(CUSTOM_RULES_PATH)

print("PAGE_MODE        =", PAGE_MODE)
print("PAGE_RANGES      =", PAGE_RANGES)
print("REVIEW_MODE      =", REVIEW_MODE)
print("RENDER_CROPS     =", RENDER_CROPS)
print("CUSTOM_RULES_PATH=", CUSTOM_RULES_PATH)


PAGE_MODE        = content
PAGE_RANGES      = {'q1': (13, None), 'q2': (3, None), 'q3': (3, None), 'q4': (4, None), 'q5': (3, None)}
REVIEW_MODE      = suspicious
RENDER_CROPS     = suspicious
CUSTOM_RULES_PATH= /kaggle/working/output_task2_dntc_ocr_review/configs/custom_ocr_rules.csv


In [9]:

from typing import Sequence, Optional, Tuple, Dict, List
from pathlib import Path
import pandas as pd

def run_extract_notebook(
    pdf_paths: Sequence[Path],
    out_dir: Path,
    page_ranges: Dict[str, Tuple[int, Optional[int]]],
    custom_rules_path: Optional[Path] = None,
    render_crops: str = "suspicious",
    crop_dpi: int = 170,
    review_mode: str = "suspicious",
    review_min_score: int = 1,
    use_underthesea: bool = True,
    enable_auto_correction: bool = True,
):
    out_dir = Path(out_dir)
    ensure_dir(out_dir)

    custom_rules = load_custom_rules(custom_rules_path)

    all_blocks = []
    pdf_lookup = {}

    for pdf_path in pdf_paths:
        pdf_path = Path(pdf_path)
        work_id = infer_work_id_from_pdf_name(pdf_path)
        start_page, end_page = page_ranges.get(work_id, (1, None))

        print(f"\n=== Extract {work_id}: {pdf_path.name} | pages {start_page} -> {end_page or 'end'} ===")
        pdf_lookup[pdf_path.name] = pdf_path

        blocks = extract_pdf_blocks(
            pdf_path,
            work_id=work_id,
            start_page=start_page,
            end_page=end_page,
        )
        print("blocks:", len(blocks))
        all_blocks.extend(blocks)

    paragraphs = blocks_to_paragraphs(
        all_blocks,
        enable_auto_correction=enable_auto_correction,
        custom_rules=custom_rules,
    )
    print("\nparagraphs:", len(paragraphs))

    sentences = paragraphs_to_sentences(
        paragraphs,
        use_underthesea=use_underthesea,
    )
    print("sentences:", len(sentences))

    add_crops_to_sentences(
        sentences,
        pdf_lookup=pdf_lookup,
        out_dir=out_dir,
        mode=render_crops,
        dpi=crop_dpi,
    )

    save_outputs(
        out_dir=out_dir,
        blocks=all_blocks,
        paras=paragraphs,
        sents=sentences,
        review_mode=review_mode,
        review_min_score=review_min_score,
    )

    print("\nDone.")
    print("Output directory:", out_dir)
    print("Review XLSX     :", out_dir / "review" / "suspicious_review.xlsx")
    print("All sentences   :", out_dir / "data" / "sentences.csv")
    print("Summary         :", out_dir / "summary.csv")

    return all_blocks, paragraphs, sentences

blocks, paragraphs, sentences = run_extract_notebook(
    pdf_paths=pdf_files,
    out_dir=OUTPUT_DIR,
    page_ranges=PAGE_RANGES,
    custom_rules_path=CUSTOM_RULES_PATH,
    render_crops=RENDER_CROPS,
    crop_dpi=CROP_DPI,
    review_mode=REVIEW_MODE,
    review_min_score=REVIEW_MIN_SCORE,
    use_underthesea=USE_UNDERTHESEA,
    enable_auto_correction=ENABLE_AUTO_CORRECTION,
)



=== Extract q1: q1.pdf | pages 13 -> end ===


extract blocks q1.pdf: 100%|██████████| 490/490 [00:35<00:00, 13.68it/s]


blocks: 4162

=== Extract q2: q2.pdf | pages 3 -> end ===


extract blocks q2.pdf: 100%|██████████| 526/526 [00:43<00:00, 12.13it/s]


blocks: 4122

=== Extract q3: q3.pdf | pages 3 -> end ===


...

blocks: 4027

=== Extract q5: q5.pdf | pages 3 -> end ===


extract blocks q5.pdf: 100%|██████████| 473/473 [00:40<00:00, 11.74it/s]


blocks: 7777

paragraphs: 20551
sentences: 34905


render crops: 100%|██████████| 12481/12481 [06:06<00:00, 34.06it/s]
/tmp/ipykernel_24/3604000572.py:297: DeprecationWarning: Call to deprecated function copy (Use copy(obj) or cell.obj = cell.obj + other).
  cell.alignment = cell.alignment.copy(wrap_text=True, vertical="top")
/tmp/ipykernel_24/3604000572.py:301: DeprecationWarning: Call to deprecated function copy (Use copy(obj) or cell.obj = cell.obj + other).
  cell.alignment = cell.alignment.copy(wrap_text=True, vertical="top")



Done.
Output directory: /kaggle/working/output_task2_dntc_ocr_review
Review XLSX     : /kaggle/working/output_task2_dntc_ocr_review/review/suspicious_review.xlsx
All sentences   : /kaggle/working/output_task2_dntc_ocr_review/data/sentences.csv
Summary         : /kaggle/working/output_task2_dntc_ocr_review/summary.csv


In [10]:

summary_path = OUTPUT_DIR / "summary.csv"
review_path = OUTPUT_DIR / "review" / "suspicious_review.xlsx"
candidates_path = OUTPUT_DIR / "review" / "spell_candidates.csv"
sentences_path = OUTPUT_DIR / "data" / "sentences.csv"

df_summary = pd.read_csv(summary_path)
df_review = pd.read_excel(review_path, sheet_name="review")
df_candidates = pd.read_csv(candidates_path)

print("Summary:")
display(df_summary)

print("Review rows:", len(df_review))
display(
    df_review.sort_values(["quality_score", "work_id", "page"], ascending=[False, True, True])
    .head(50)
)

print("Top spell candidates:")
display(df_candidates.head(100))


Summary:


,work_id,pages,blocks,paragraphs,sentences,suspicious_sentences,high_suspicious_score_ge_5,auto_rule_sentences,avg_quality_score,max_quality_score
0,q1,490,4162,3037,6378,1370,106,158,0.506,11
1,q2,518,4122,2935,5263,1098,88,305,0.496,9
2,q3,542,3915,3217,7014,3600,618,982,1.535,12
3,q4,498,4027,2828,6233,1460,99,568,0.551,11
4,q5,466,7777,8534,10017,4953,337,406,1.428,11


Review rows: 12481


,work_id,pdf_file,page,para_id,sent_id,type,quality_score,reasons,suggestions,correction_rules,raw_text,clean_text,auto_text,corrected_text_manual,review_status,final_text,crop_path,bbox,block_ids
0,q3,q3.pdf,36,0,437,sentence,12,junk_chars:_; glued_number_letter; high_no_acc...,TINH -> TỈNH | PHU -> PHỦ,body_huyen_low,"._+. S6ng La Tinh: ở huyén Phu; Cát, sông rộng...","._+. S6ng La Tinh: ở huyén Phu; Cát, sông rộng...","S6ng La Tinh: ở huyện Phu; Cát, sông rộng 16 t...","S6ng La Tinh: ở huyện Phu; Cát, sông rộng 16 t...",NaN,"S6ng La Tinh: ở huyện Phu; Cát, sông rộng 16 t...",crops/q3/q3_p0036_b0_bc904d1fbc.png,"[42.80619812011719, 43.172523498535156, 327.49...",[0]
1,q3,q3.pdf,534,3,6867,sentence,12,junk_chars:_|; unexpected_unicode; high_no_acc...,Nim -> Năm,NaN,"- Nguyễn Thị Duệ: người huyện Chí Linh, thông ...","- Nguyễn Thị Duệ: người huyện Chí Linh, thông ...",trong Chi |Linh Phong thổ kd).,trong Chi |Linh Phong thổ kd).,NaN,trong Chi |Linh Phong thổ kd).,crops/q3/q3_p0534_b1-2-3-4_a03bdfc2f3.png,"[5.5518951416015625, 121.90678405761719, 305.1...","[1, 2, 3, 4]"
2,q1,q1.pdf,502,7,6375,sentence,11,junk_chars:|; unexpected_unicode; high_no_acce...,NHAT -> NHẤT | THONG -> THỐNG,title_nhat_thong,NHAT THONG (Hi | NHẤT THONG CHÍ,NHAT THONG (Hi | NHẤT THONG CHÍ,NHAT THONG (Hi | NHẤT THỐNG CHÍ,NHAT THONG (Hi | NHẤT THỐNG CHÍ,NaN,NHAT THONG (Hi | NHẤT THỐNG CHÍ,crops/q1/q1_p0502_b2_8396ec8be9.png,"[196.2115478515625, 126.72000122070312, 232.88...",[2]
3,q3,q3.pdf,36,2,442,sentence,11,junk_chars:_; unexpected_unicode; glued_number...,dim -> dặm,body_dam_after_num,Có ba nguồn: Một nguồn từ mii Phong Sơn chả y ...,Có ba nguồn: Một nguồn từ mii Phong Sơn chả y ...,Có ba nguồn: Một nguồn từ mii Phong Sơn chả y ...,Có ba nguồn: Một nguồn từ mii Phong Sơn chả y ...,NaN,Có ba nguồn: Một nguồn từ mii Phong Sơn chả y ...,crops/q3/q3_p0036_b1-2-3-4-5_bea2c8565b.png,"[29.308761596679688, 161.2831573486328, 324.09...","[1, 2, 3, 4, 5]"
4,q3,q3.pdf,105,0,1309,sentence,11,junk_chars:|€; unexpected_unicode; missing_spa...,dim -> dặm | bi€n -> biển,body_bien_euro,"ở ven bi€n làm nghề chài lưới,. dân ở ven núi ...","ở ven bi€n làm nghề chài lưới,. dân ở ven núi ...","Các tết Nguyên đán, Đoan dượng, Tam nguyên... ...","Các tết Nguyên đán, Đoan dượng, Tam nguyên... ...",NaN,"Các tết Nguyên đán, Đoan dượng, Tam nguyên... ...",crops/q3/q3_p0105_b0_2727f39a89.png,"[7.9558424949646, 33.92854309082031, 299.68246...",[0]
5,q3,q3.pdf,263,2,3444,sentence,11,junk_chars:_|~ˆ; high_no_accent_ratio:0.44; mi...,NaN,NaN,ˆ Hoàng Tế Mr người xã Đông Ngạc huyện T ừ ¡ ê...,ˆ Hoàng Tế Mr người xã Đông Ngạc huyện T ừ ¡ ê...,Thai hậu oi hành an Thái Tur cũ Là,Thai hậu oi hành an Thái Tur cũ Là,NaN,Thai hậu oi hành an Thái Tur cũ Là,crops/q3/q3_p0263_b3-4-5-6-7-8_d53d12a9e6.png,"[4.02174186706543, 159.06826782226562, 305.613...","[3, 4, 5, 6, 7, 8]"
6,q3,q3.pdf,267,0,3497,sentence,11,junk_chars:|; unexpected_unicode; glued_number...,NGUYEN -> NGUYỄN,NaN,por NGUYEN Lê Thị Bản: người x xã Hoang Phúc: ...,por NGUYEN Lê Thị Bản: người x xã Hoang Phúc: ...,por NGUYEN Lê Thị Bản: người x xã Hoang Phúc: ...,por NGUYEN Lê Thị Bản: người x xã Hoang Phúc: ...,NaN,por NGUYEN Lê Thị Bản: người x xã Hoang Phúc: ...,crops/q3/q3_p0267_b0-1-2-3_7d4c2b3892.png,"[9.539993286132812, 33.293113708496094, 315.17...","[0, 1, 2, 3]"
7,q3,q3.pdf,375,3,4858,sentence,11,junk_chars:|; unexpected_unicode; auto_rules_a...,NHAT -> NHẤT | TINH -> TỈNH | dim -> dặm | dam...,body_dam_after_num,Huyện Thư TH! § cách phủ 25 dam về phía tây bắ...,Huyện Thư TH! § cách phủ 25 dam về phía tây bắ...,"Bin triều năm Tự Đức thứ 4, bồ tri huyện đo ph...","Bin triều năm Tự Đức thứ 4, bồ tri huyện đo ph...",NaN,"Bin triều năm Tự Đức thứ 4, bồ tri huyện đo ph...",crops/q3/q3_p0375_b3-4-5_011d82b763.png,"[54.36000061035156, 220.72947692871094, 335.88...","[3, 4, 5]"
8,q3,q3.pdf,433,1,5626,sentence,11,junk_chars:_~; unexpected_unicode; glued_numbe...,TINH -> TỈNH,body_tinh_place,"- đặt. các; chức. trấn thi, hiệp. trấn,

Top spell candidates:


,token,count,suggestion,examples
0,nam,7539,NaN,q1:p16: Những địa điểm đóng dinh đều là đất Th...
1,Thanh,3410,NaN,q1:p17: Bốn phía trên mặt thành xây 24 pháo đà...
2,trong,3195,NaN,"q1:p13: Xem sự ghi chép trong các sách, thì th..."
3,cho,3146,NaN,"q1:p14: chúa Chiêm Thành là Chế Củ đem về, Chế..."
4,hai,3033,NaN,"q1:p13: _Xét: hai xứ Thuận Quảng, đời Hán] là ..."
...,...,...,...,...
95,Theo,296,NaN,"q1:p98: Theo thiên văn, thuộc về khu vực sao D..."
96,non,294,NaN,"q1:p13: Kinh sư là nơi miền núi, miền biến đều..."
97,man,293,NaN,"q1:p116: Đầu bản triều vẫn theo như thế, cũng ..."
98,danh,292,NaN,q1:p37: Phía đông và phía tây dựng Đông vu và ...


In [11]:

# Xem nhanh một số crop ảnh scan của các câu nghi lỗi.
from IPython.display import display, Image

N_SHOW_CROPS = 12
if "crop_path" in df_review.columns:
    sample = df_review.dropna(subset=["crop_path"]).head(N_SHOW_CROPS)
    print("Số crop hiển thị:", len(sample))
    for _, row in sample.iterrows():
        print("\n", row.get("record_id"), "|", row.get("work_id"), "page", row.get("page"))
        print("raw :", row.get("raw_text"))
        print("auto:", row.get("auto_text"))
        crop_path = row.get("crop_path")
        if isinstance(crop_path, str) and Path(crop_path).exists():
            display(Image(filename=crop_path))
else:
    print("Không có cột crop_path. Kiểm tra RENDER_CROPS.")


Số crop hiển thị: 12

 None | q3 page 36
raw : ._+. S6ng La Tinh: ở huyén Phu; Cát, sông rộng 16 trượng, nguồn. từ phía đông nam núi Hội Sơn, qua hai ấp Yên Long và Hợp Long, lại chảy về phía đông 35 dặm đến ấp Xuân Hội, làm sông La Tinh.
auto: S6ng La Tinh: ở huyện Phu; Cát, sông rộng 16 trượng, nguồn.

 None | q3 page 534
raw : - Nguyễn Thị Duệ: người huyện Chí Linh, thông | minh hơn người, hoe rộng văn hay; nim mdi hơn 10 tuổi, cải trang làm con trai để đi Bọc; khi lớn; ứng thí khoa thị Hội đời Mạc, đỗ Tiến sĩ: Vua Mạc thấy dung mạo, giống con gái, hỏi ga mới biết, lấy làm la. Đến khi nha Mạc mất, ẩn ở din gian, vua Lê _nghe tiếng, cho triệu vào cung để dạy cụng nữ, cho “ hiệu là Nghi Ái quan, dùng văn chương hầu ha,. không rời tả hữu; mỗi khi vua hỏi việc gì, thị liền dùng sự tích xưa. nay chép trong kinh sử để đáp, vua Lê: khen ngợi, cấp. cho các thuế ở bản xã làm ngụ lộc. Đến năm 70 tuổi xin về làng, dựng am Đàm Hoa để ở. Nay xã Kiệt Đặc thờ làm thần, vẫn còn _ bì kí. (việc chép.

In [12]:

# Thêm rule nhanh ngay trong notebook. Sau khi chạy cell này, quay lại chạy cell extract ở mục 4.
# Không nên thêm rule quá rộng. Ưu tiên sửa lỗi chắc chắn/lặp lại nhiều lần.

NEW_RULES = [
    # Ví dụ:
    # {"pattern": r"\bDai Nam\b", "replacement": "Đại Nam", "scope": "body", "name": "custom_dai_nam"},
    # {"pattern": r"\bPHAN DA\b", "replacement": "PHẦN DÃ", "scope": "heading", "name": "custom_phan_da"},
]

if NEW_RULES:
    import csv
    file_exists = CUSTOM_RULES_PATH.exists()
    with CUSTOM_RULES_PATH.open("a", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["pattern", "replacement", "scope", "name"])
        if not file_exists:
            writer.writeheader()
        for r in NEW_RULES:
            writer.writerow(r)
    print("Đã append rule vào:", CUSTOM_RULES_PATH)
else:
    print("NEW_RULES đang rỗng. Thêm rule rồi chạy lại cell này nếu cần.")


NEW_RULES đang rỗng. Thêm rule rồi chạy lại cell này nếu cần.


In [13]:

# Mặc định dùng chính file review trong OUTPUT_DIR.
# Nếu upload file đã sửa vào Kaggle input, đổi path tại đây.
EDITED_REVIEW_FILE = OUTPUT_DIR / "review" / "suspicious_review.xlsx"

final_sentences = apply_manual_review(
    out_dir=OUTPUT_DIR,
    review_path=EDITED_REVIEW_FILE,
)

final_csv = OUTPUT_DIR / "final" / "final_sentences.csv"
final_jsonl = OUTPUT_DIR / "final" / "final_sentences.jsonl"
final_text_dir = OUTPUT_DIR / "final" / "texts"

print("Final CSV  :", final_csv)
print("Final JSONL:", final_jsonl)
print("Final texts:", final_text_dir)

df_final = pd.read_csv(final_csv)
display(df_final.head(30))


Final CSV  : /kaggle/working/output_task2_dntc_ocr_review/final/final_sentences.csv
Final JSONL: /kaggle/working/output_task2_dntc_ocr_review/final/final_sentences.jsonl
Final texts: /kaggle/working/output_task2_dntc_ocr_review/final/texts


,work_id,pdf_file,page,para_id,sent_id,type,quality_score,reasons,suggestions,correction_rules,raw_text,clean_text,auto_text,corrected_text_manual,review_status,final_text,crop_path,bbox,block_ids
0,q1,q1.pdf,13,0,0,heading,5,auto_rules_applied; large_auto_change:0.60,DAI -> ĐẠI | THONG -> THỐNG,title_dai_nam,DAI NAM NHẬT THONG CHÍ,DAI NAM NHẬT THONG CHÍ,ĐẠI NAM NHẤT THỐNG CHÍ,ĐẠI NAM NHẤT THỐNG CHÍ,NaN,ĐẠI NAM NHẤT THỐNG CHÍ,crops/q1/q1_p0013_b0_33418b6501.png,"[73.37679290771484, 102.70913696289062, 313.60...",[0]
1,q1,q1.pdf,13,1,1,sentence,7,junk_chars:|; unexpected_unicode; high_no_acce...,NaN,NaN,| QUYỀN I,| QUYỀN I,| QUYỀN I,| QUYỀN I,NaN,| QUYỀN I,crops/q1/q1_p0013_b1_f2157e415a.png,"[161.4091339111328, 139.91036987304688, 220.40...",[1]
2,q1,q1.pdf,13,2,2,heading,5,heading_no_accent_ratio:0.50; auto_rules_appli...,NaN,heading_kinh_su,KINH SU,KINH SU,KINH SƯ,KINH SƯ,NaN,KINH SƯ,crops/q1/q1_p0013_b2_d064588bb2.png,"[164.7501220703125, 166.60684204101562, 227.50...",[2]
3,q1,q1.pdf,13,3,3,sentence,1,auto_rules_applied,NaN,body_phia,"Kinh sư là nơi miền núi, miền biến đều họp về,...","Kinh sư là nơi miền núi, miền biến đều họp về,...","Kinh sư là nơi miền núi, miền biến đều họp về,...","Kinh sư là nơi miền núi, miền biến đều họp về,...",NaN,"Kinh sư là nơi miền núi, miền biến đều họp về,...",crops/q1/q1_p0013_b3-4_d9d9fb3f79.png,"[55.79338073730469, 189.01304626464844, 344.17...","[3, 4]"
4,q1,q1.pdf,13,3,4,sentence,2,auto_rules_applied,phia -> phía,body_phia,"Đường thủy thì có cửa Thuận An, cửa Tư Hiền sâ...","Đường thủy thì có cửa Thuận An, cửa Tư Hiền sâ...","Đường thủy thì có cửa Thuận An, cửa Tư Hiền sâ...","Đường thủy thì có cửa Thuận An, cửa Tư Hiền sâ...",NaN,"Đường thủy thì có cửa Thuận An, cửa Tư Hiền sâ...",crops/q1/q1_p0013_b3-4_d9d9fb3f79.png,"[55.79338073730469, 189.01304626464844, 344.17...","[3, 4]"
5,q1,q1.pdf,13,3,5,sentence,1,auto_rules_applied,NaN,body_phia,"Xem sự ghi chép trong các sách, thì thấy: từ đ...","Xem sự ghi chép trong các sách, thì thấy: từ đ...","Xem sự ghi chép trong các sách, thì thấy: từ đ...","Xem sự ghi chép trong các sách, thì thấy: từ đ...",NaN,"Xem sự ghi chép trong các sách, thì thấy: từ đ...",crops/q1/q1_p0013_b3-4_d9d9fb3f79.png,"[55.79338073730469, 189.01304626464844, 344.17...","[3, 4]"
6,q1,q1.pdf,13,4,6,sentence,4,junk_chars:_; unexpected_unicode,NaN,NaN,"_Xét: hai xứ Thuận Quảng, đời Hán] là huyện Tư...","_Xét: hai xứ Thuận Quảng, đời Hán] là huyện Tư...","_Xét: hai xứ Thuận Quảng, đời Hán] là huyện Tư...","_Xét: hai xứ Thuận Quảng, đời Hán] là huyện Tư...",NaN,"_Xét: hai xứ Thuận Quảng, đời Hán] là huyện Tư...",crops/q1/q1_p0013_b5_d07890fa4d.png,"[56.974884033203125, 363.1006774902344, 335.92...",[5]
7,q1,q1.pdf,13,4,7,sentence,0,NaN,NaN,NaN,"Đường thư, Địa lý chí chép: giáp Châu Hoành Sơ...","Đường thư, Địa lý chí chép: giáp Châu Hoành Sơ...","Đường thư, Địa lý chí chép: giáp Châu Hoành Sơ...",NaN,NaN,"Đường thư, Địa lý chí chép: giáp Châu Hoành Sơ...",NaN,"[56.974884033203125, 363.1006774902344, 335.92...",[5]
8,q1,q1.pdf,13,4,8,sentence,0,NaN,NaN,NaN,"Đời Lý Thánh Tông, năm Thiên huống Bảo Tượng t...","Đời Lý Thánh Tông, năm Thiên huống Bảo Tượng t...","Đời Lý Thánh Tông, năm Thiên huống Bảo Tượng t...",NaN,NaN,"Đời Lý Thánh Tông, năm Thiên huống Bảo Tượng t...",NaN,"[56.974884033203125, 363.1006774902344, 335.92...",[5]
9,q1,q1.pdf,14,0,9,sentence,0,NaN,NaN,NaN,"chúa Chiêm Thành là Chế Củ đem về, Chế Củ xin ...","chúa Chiêm Thành là Chế Củ đem về, Chế Củ xin ...","chúa Chiêm Thành là Chế Củ đem về, Chế Củ xin ...",NaN,NaN,"chúa Chiêm Thành là Chế Củ đem về, Chế Củ xin ...",NaN,"[85.98634338378906, 49.229454040527344, 364.72...",[0]


In [14]:

ENABLE_HF_SUGGESTIONS = False
HF_MODEL_NAME = "bmd1905/vietnamese-correction-v2"
HF_MAX_ROWS = 300

if ENABLE_HF_SUGGESTIONS:
    import torch
    from transformers import pipeline

    device = 0 if torch.cuda.is_available() else -1
    print("Loading HF model:", HF_MODEL_NAME, "device:", device)
    corrector = pipeline("text2text-generation", model=HF_MODEL_NAME, device=device)

    df_hf = df_review.copy()
    texts = df_hf["auto_text"].fillna(df_hf["raw_text"]).astype(str).tolist()
    texts = texts[:HF_MAX_ROWS]

    suggestions = []
    for i, text in enumerate(texts):
        try:
            out = corrector(text, max_new_tokens=256, num_beams=4)
            sug = out[0].get("generated_text", "").strip()
        except Exception as e:
            sug = ""
            print("HF failed row", i, e)
        suggestions.append(sug)

    df_hf.loc[:len(suggestions)-1, "hf_suggestion"] = suggestions

    hf_out = OUTPUT_DIR / "review" / "suspicious_review_with_hf_suggestions.xlsx"
    write_excel(hf_out, {"review": df_hf})
    print("Wrote:", hf_out)
    display(df_hf[["record_id", "raw_text", "auto_text", "hf_suggestion", "quality_score", "reasons"]].head(50))
else:
    print("HF suggestions đang tắt. Đổi ENABLE_HF_SUGGESTIONS = True nếu cần.")


HF suggestions đang tắt. Đổi ENABLE_HF_SUGGESTIONS = True nếu cần.


In [15]:

import shutil
zip_base = OUTPUT_DIR.parent / f"{OUTPUT_DIR.name}_export"
zip_path = shutil.make_archive(str(zip_base), "zip", OUTPUT_DIR)
print("ZIP:", zip_path)


ZIP: /kaggle/working/output_task2_dntc_ocr_review_export.zip
